### Healthcare Readmission Analytics

#### Notebook 04 — Feature Engineering & Machine Learning Preparation

#### Purpose
#####
This notebook prepares the cleaned healthcare dataset for machine learning.

The objectives are:

- create modeling features
- prepare target variables
- encode categorical variables
- remove non-predictive identifiers
- create a machine-learning-ready dataset

#### Why This Matters

Machine learning models cannot directly use raw healthcare data.

Feature engineering transforms raw variables into meaningful predictors that improve model performance.

Import Libraries

In [5]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

Load Clean Dataset

In [6]:
# ============================================
# LOAD DATA
# ============================================

df = pd.read_csv(
    "../data/processed/cleaned_healthcare_data.csv"
)

print(df.shape)

(101766, 47)


Create Target Variable

Current target:
NO
- >30
- <30

Machine learning classification works better with a binary target initially.

Create Binary Readmission Target

Patients are classified into:

- 1 = Readmitted
- 0 = Not Readmitted

This simplifies the prediction problem and aligns with many hospital readmission initiatives.

In [7]:
# ============================================
# TARGET VARIABLE
# ============================================

df["readmitted_flag"] = df["readmitted"].apply(
    lambda x: 0 if x == "NO" else 1
)

df["readmitted_flag"].value_counts()

readmitted_flag
0    54864
1    46902
Name: count, dtype: int64

Create Healthcare Utilization Feature

Why We Do This -

Instead of:

- number_outpatient
- number_emergency
- number_inpatient

separately,

we create a total utilization score.

Hospitals often use utilization history to assess risk.

In [8]:
# ============================================
# TOTAL HEALTHCARE UTILIZATION
# ============================================

df["total_utilization"] = (

    df["number_outpatient"]

    + df["number_emergency"]

    + df["number_inpatient"]

)

df["total_utilization"].describe()

count    101766.000000
mean          1.202759
std           2.291781
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max          80.000000
Name: total_utilization, dtype: float64

Create Medication Complexity Feature

Medication count is already useful.

Let's explicitly represent complexity.

In [9]:
# ============================================
# MEDICATION COMPLEXITY
# ============================================

df["high_medication_count"] = (
    df["num_medications"] >= 15
).astype(int)

df["high_medication_count"].value_counts()

high_medication_count
1    52313
0    49453
Name: count, dtype: int64

Create Length of Stay Category

Hospitals often classify patients by stay duration.

In [10]:
# ============================================
# LENGTH OF STAY CATEGORY
# ============================================

df["los_category"] = pd.cut(

    df["time_in_hospital"],

    bins=[0,3,7,14],

    labels=[
        "Short Stay",
        "Medium Stay",
        "Long Stay"
    ]

)

df["los_category"].value_counts()

los_category
Short Stay     49188
Medium Stay    37288
Long Stay      15290
Name: count, dtype: int64

Create Age Numeric Feature

Why we do this: 

Current age values:

[70-80)

Machine learning cannot use these directly.

In [11]:
# ============================================
# AGE TO NUMERIC
# ============================================

age_mapping = {

    "[0-10)":5,
    "[10-20)":15,
    "[20-30)":25,
    "[30-40)":35,
    "[40-50)":45,
    "[50-60)":55,
    "[60-70)":65,
    "[70-80)":75,
    "[80-90)":85,
    "[90-100)":95

}

df["age_numeric"] = (
    df["age"].map(age_mapping)
)

df["age_numeric"].head()

0     5
1    15
2    25
3    35
4    45
Name: age_numeric, dtype: int64

Remove Identifiers

Why We Remove Them

These columns identify records.

They do not help predict readmission.

In [12]:
# ============================================
# IDENTIFIER COLUMNS
# ============================================

identifier_columns = [

    "encounter_id",
    "patient_nbr"

]

identifier_columns

['encounter_id', 'patient_nbr']

Select Modeling Features

For now, create a candidate feature list.

In [13]:
candidate_features = [

    "age_numeric",

    "time_in_hospital",

    "num_lab_procedures",

    "num_procedures",

    "num_medications",

    "number_outpatient",

    "number_emergency",

    "number_inpatient",

    "number_diagnoses",

    "total_utilization",

    "high_medication_count"

]

candidate_features

['age_numeric',
 'time_in_hospital',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'number_diagnoses',
 'total_utilization',
 'high_medication_count']

Create Modeling Dataset

In [14]:
# ============================================
# MODELING DATASET
# ============================================

modeling_df = df[
    candidate_features +
    ["readmitted_flag"]
].copy()

print(modeling_df.shape)

modeling_df.head()

(101766, 12)


,age_numeric,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,total_utilization,high_medication_count,readmitted_flag
0,5,1,41,0,1,0,0,0,1,0,0,0
1,15,3,59,0,18,0,0,0,9,0,1,1
2,25,2,11,5,13,2,0,1,6,3,0,0
3,35,2,44,1,16,0,0,0,7,0,1,0
4,45,1,51,0,8,0,0,0,5,0,0,0


Save Modeling Dataset

The final modeling dataset will be used in the machine learning notebook.

In [15]:
# ============================================
# SAVE MODELING DATASET
# ============================================

modeling_df.to_csv(

    "../data/processed/modeling_dataset.csv",

    index=False

)

print("Modeling dataset saved.")

Modeling dataset saved.


Verification

In [16]:
test_modeling = pd.read_csv(
    "../data/processed/modeling_dataset.csv"
)

print(test_modeling.shape)

(101766, 12)


11 features + 1 target.

In [17]:
modeling_df.shape

(101766, 12)

In [18]:
modeling_df.head()

,age_numeric,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,total_utilization,high_medication_count,readmitted_flag
0,5,1,41,0,1,0,0,0,1,0,0,0
1,15,3,59,0,18,0,0,0,9,0,1,1
2,25,2,11,5,13,2,0,1,6,3,0,0
3,35,2,44,1,16,0,0,0,7,0,1,0
4,45,1,51,0,8,0,0,0,5,0,0,0


In [19]:
df["high_medication_count"].value_counts()

high_medication_count
1    52313
0    49453
Name: count, dtype: int64

In [ ]:
df["los_category"].value_counts()

los_category
Short Stay     49188
Medium Stay    37288
Long Stay      15290
Name: count, dtype: int64

Dataset Structure

| Type          | Count |
| ------------- | ----: |
| Features      |    11 |
| Target        |     1 |
| Total Columns |    12 |

Healthcare Utilization Feature

total_utilization = number_outpatient + number_emergency + number_inpatient

This is a strong healthcare feature because it captures overall healthcare engagement.

Hospitals frequently use utilization history when identifying high-risk patients.

High Medication Count Feature

| Value                     |  Count |
| ------------------------- | -----: |
| High Medication Count = 1 | 52,313 |
| High Medication Count = 0 | 49,453 |

Interpretation

Very balanced feature.

This is excellent for machine learning.

Healthcare Meaning

Patients taking: 15 or more medications are being treated as a higher-complexity population.

This often correlates with:

- chronic disease burden
- polypharmacy
- higher readmission risk

Length of Stay Categories

| Category    |  Count |
| ----------- | -----: |
| Short Stay  | 49,188 |
| Medium Stay | 37,288 |
| Long Stay   | 15,290 |

Interpretation

Most hospitalizations are short or medium stays.

Only a smaller subset experiences longer admissions.

Healthcare Meaning

Longer stays often indicate:

- more severe illness
- more complex care
- greater resource utilization



In [21]:
# ============================================
# SAVE FEATURE LIST
# ============================================

feature_list = pd.DataFrame({
    "feature_name": candidate_features
})

feature_list.to_csv(
    "../outputs/model_feature_list.csv",
    index=False
)

feature_list

,feature_name
0,age_numeric
1,time_in_hospital
2,num_lab_procedures
3,num_procedures
4,num_medications
5,number_outpatient
6,number_emergency
7,number_inpatient
8,number_diagnoses
9,total_utilization
